# C318 - Trabalho 2 - Notebook 02: TPOT oficial

Este notebook executa a ferramenta TPOT 1.1.0 sobre a mesma base filtrada e a mesma partição 60/20/20 da referência manual. A ferramenta recebe somente o treino; validação e teste permanecem externos.

O notebook nasce com todas as execuções caras desligadas. Antes de ativar cada bloco, confira o protocolo e mantenha os resultados já obtidos. Resultados oficiais serão salvos em resultados/tpot/.


## 0. Protocolo declarado antes dos resultados

- Budgets: 5, 30 e 60 minutos; sementes: 42, 101, 223, 347 e 409.
- Otimização interna: F1 macro, CV estratificada de 5 folds somente no treino.
- Entrada: mesmas colunas descartadas pelo Trabalho 1; idade em anos e os dois atributos de pico são transformações determinísticas por registro. Ausências recebem sentinelas fixos, sem estimar mediana ou frequência fora das pipelines do TPOT.
- Codificação categórica: primeira etapa de cada pipeline candidata do TPOT, portanto ajustada dentro dos folds internos. Não usar preprocessing=True: a documentação do TPOT 1.1.0 informa que essa opção ajusta o pré-processamento no treino inteiro antes de sua CV.
- Escolha de budget: maior média de F1 macro na validação; em empate, menor budget. O teste só será usado depois que essa escolha for salva.


In [ ]:
RUN_VALIDATION = False
RUN_DERIVED_ABLATION = False
RUN_IMPORTANCE = False
RUN_FINAL_TEST = False  # Ativar somente após congelar o budget em budget_selecionado.json

BUDGETS_MIN = [5, 30, 60]
SEEDS_TPOT = [42, 101, 223, 347, 409]
FINAL_SEED = 42

assert not (RUN_FINAL_TEST and RUN_VALIDATION), "Não misture escolha de budget e consulta ao teste."


## 1. Ambiente, dados e asserções

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import tpot
from sklearn.inspection import permutation_importance

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "HPLC_data.csv").exists():
            return p
    raise FileNotFoundError("Abra o notebook a partir do repositório clonado.")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))

from c318_common import SEED, SEEDS, TARGET, compute_metrics, hardware_info, load_split
from tpot_protocol import (
    prepare_tpot_input, align_numeric_inputs, build_tpot, fit_measure_predict, pipeline_repr,
    select_budget, write_json,
)

RESULTS = ROOT / "resultados" / "tpot"
CHECKPOINTS = RESULTS / "checkpoints"
PREDICTIONS = RESULTS / "predicoes_validacao"
PIPELINES = RESULTS / "pipelines"
for folder in [RESULTS, CHECKPOINTS, PREDICTIONS, PIPELINES]:
    folder.mkdir(parents=True, exist_ok=True)

assert set(SEEDS_TPOT).issubset(set(SEEDS))
print("Raiz:", ROOT)
print("TPOT:", tpot.__version__)
print("Hardware:", hardware_info())


In [ ]:
train_df, val_df, test_df = load_split(ROOT)
X_train, y_train = prepare_tpot_input(train_df, with_derived=True)
X_val, y_val = prepare_tpot_input(val_df, with_derived=True)
X_test, y_test = prepare_tpot_input(test_df, with_derived=True)
X_train, X_val, X_test = align_numeric_inputs(X_train, X_val, X_test)
LABELS = np.sort(y_train.unique())

assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
assert y_train.nunique() == y_val.nunique() == y_test.nunique() == 8
assert X_train.columns.equals(X_val.columns) and X_train.columns.equals(X_test.columns)
print("Treino, validação, teste:", X_train.shape, X_val.shape, X_test.shape)
print(y_train.value_counts())


## 2. Execuções de validação

Cada execução ajusta TPOT apenas em X_train e y_train. A validação é usada somente depois de fit. Os resultados, predições, metadados e representação da pipeline são persistidos após cada seed para que uma falha não apague os trabalhos concluídos.

O checkpoint do TPOT é ignorado pelo Git porque pode ser grande; as métricas, as predições e a pipeline textual permanecem versionáveis.


In [ ]:
VALIDATION_CSV = RESULTS / "metricas_validacao_runs.csv"

if RUN_VALIDATION:
    old = (pd.read_csv(VALIDATION_CSV)
           if VALIDATION_CSV.exists() and VALIDATION_CSV.stat().st_size > 0
           else pd.DataFrame())
    existing = set(zip(old.get("budget_min", []), old.get("seed", [])))
    new_rows = []

    for budget in BUDGETS_MIN:
        for seed in SEEDS_TPOT:
            if (budget, seed) in existing:
                print(f"Já registrado: budget={budget}, seed={seed}")
                continue
            run_id = f"budget_{budget:02d}m_seed_{seed}"
            checkpoint = CHECKPOINTS / run_id
            metadata = {
                "run_id": run_id, "budget_min": budget, "seed": seed,
                "subset_fit": "train", "subset_eval": "val",
                "target": TARGET, "tpot_version": tpot.__version__,
                "started_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            try:
                model = build_tpot(X_train, y_train, budget, seed, checkpoint)
                metrics, pred = fit_measure_predict(model, X_train, y_train, X_val, y_val, LABELS)
                row = {"sistema": "TPOT", "versao": tpot.__version__, "run_id": run_id,
                       "budget_min": budget, "seed": seed, "subconjunto": "val", **metrics}
                pred.insert(0, "sl_no", val_df.loc[pred.index, "Sl No"].to_numpy())
                pred.to_csv(PREDICTIONS / f"{run_id}.csv", index_label="row_index")
                (PIPELINES / f"{run_id}.txt").write_text(pipeline_repr(model), encoding="utf-8")
                metadata.update({"status": "ok", "metrics": metrics, "pipeline_file": f"pipelines/{run_id}.txt"})
                new_rows.append(row)
                print(run_id, "F1 macro:", f"{metrics['f1_macro']:.4f}")
            except Exception as exc:
                metadata.update({"status": "erro", "erro": f"{type(exc).__name__}: {exc}"})
                write_json(RESULTS / "falhas" / f"{run_id}.json", metadata)
                print(run_id, "falhou e será ignorado; consulte o JSON em resultados/tpot/falhas")
                continue
            finally:
                write_json(RESULTS / "metadados" / f"{run_id}.json", metadata)
                partial = pd.concat([old, pd.DataFrame(new_rows)], ignore_index=True)
                partial.to_csv(VALIDATION_CSV, index=False)
else:
    print("RUN_VALIDATION=False. Nenhuma busca TPOT será iniciada.")

validation_runs = pd.read_csv(VALIDATION_CSV) if VALIDATION_CSV.exists() else pd.DataFrame()
display(validation_runs)


## 3. Curva budget versus desempenho e seleção congelada

In [ ]:
SELECTION_JSON = RESULTS / "budget_selecionado.json"
SUMMARY_CSV = RESULTS / "resumo_budgets_validacao.csv"

if not validation_runs.empty:
    coverage = validation_runs.groupby("budget_min")["seed"].nunique()
    expected_complete = set(BUDGETS_MIN) <= set(coverage.index) and (coverage.reindex(BUDGETS_MIN) == len(SEEDS_TPOT)).all()
    if expected_complete:
        summary, chosen_budget = select_budget(validation_runs)
        summary.to_csv(SUMMARY_CSV, index=False)
        write_json(SELECTION_JSON, {
            "criterio": "maior média de F1 macro na validação; empate pelo menor budget",
            "budget_selecionado_min": chosen_budget,
            "seeds": SEEDS_TPOT,
            "resumo": summary.to_dict(orient="records"),
        })
        display(summary)
        print("Budget selecionado:", chosen_budget, "minutos")
    else:
        print("Validação parcial salva. Não há seleção de budget antes das 15 execuções.")
        display(coverage.rename("sementes_concluidas"))
else:
    print("A curva será criada após completar as 15 execuções de validação.")


## 4. Experimento obrigatório com atributo derivado

O enunciado pede uma execução de 30 minutos com e sem os atributos HbS_peak_presente e HbD_peak_presente. Este bloco usa a mesma validação externa e não consulta o teste. Para repetir as cinco sementes, mantenha a lista SEEDS_TPOT; caso a equipe documente outra decisão, altere-a antes de executar.


In [ ]:
ABLATION_CSV = RESULTS / "ablacao_atributo_derivado.csv"

if RUN_DERIVED_ABLATION:
    old_ablation = pd.read_csv(ABLATION_CSV) if ABLATION_CSV.exists() and ABLATION_CSV.stat().st_size > 0 else pd.DataFrame()
    rows = old_ablation.to_dict(orient="records")
    for derived_name, with_derived in [("com", True), ("sem", False)]:
        X_tr_d, y_tr_d = prepare_tpot_input(train_df, with_derived=with_derived)
        X_va_d, y_va_d = prepare_tpot_input(val_df, with_derived=with_derived)
        X_tr_d, X_va_d = align_numeric_inputs(X_tr_d, X_va_d)
        for seed in SEEDS_TPOT:
            run_id = f"ablacao_{derived_name}_30m_seed_{seed}"
            if any(r.get("atributo_derivado") == derived_name and r.get("seed") == seed for r in rows):
                print(f"Já registrado: {run_id}")
                continue
            model = build_tpot(X_tr_d, y_tr_d, 30, seed, CHECKPOINTS / run_id)
            metrics, _ = fit_measure_predict(model, X_tr_d, y_tr_d, X_va_d, y_va_d, LABELS)
            rows.append({"atributo_derivado": derived_name, "budget_min": 30, "seed": seed, **metrics})
            print(run_id, f"F1 macro: {metrics['f1_macro']:.4f}")
            pd.DataFrame(rows).to_csv(ABLATION_CSV, index=False)
    pd.DataFrame(rows).to_csv(ABLATION_CSV, index=False)
else:
    print("RUN_DERIVED_ABLATION=False. Nenhuma ablação será iniciada.")


## 5. Interpretabilidade por permutação

Depois de congelar o budget, reajuste uma vez no treino com a seed final e calcule importância por permutação somente na validação. O artefato usa os atributos de entrada do TPOT; não consulta o teste.


In [ ]:
IMPORTANCE_CSV = RESULTS / "importancia_permutacao_validacao.csv"

if RUN_IMPORTANCE:
    if not SELECTION_JSON.exists():
        raise FileNotFoundError("Execute e congele primeiro a seleção do budget.")
    chosen_budget = json.loads(SELECTION_JSON.read_text(encoding="utf-8"))["budget_selecionado_min"]
    champion = build_tpot(X_train, y_train, chosen_budget, FINAL_SEED,
                           CHECKPOINTS / f"importancia_{chosen_budget}m_seed_{FINAL_SEED}")
    champion.fit(X_train, y_train)
    pi = permutation_importance(champion, X_val, y_val, scoring="f1_macro",
                                n_repeats=10, random_state=SEED, n_jobs=1)
    importance = pd.DataFrame({"atributo": X_val.columns,
                               "queda_f1_macro_media": pi.importances_mean,
                               "queda_f1_macro_dp": pi.importances_std}).sort_values(
                                   "queda_f1_macro_media", ascending=False)
    importance.to_csv(IMPORTANCE_CSV, index=False)
    display(importance)
else:
    print("RUN_IMPORTANCE=False. A importância será executada depois da seleção.")


## 6. Teste final único

Só ative este bloco depois que a escolha do budget estiver congelada em budget_selecionado.json e revisada pela equipe. Ele bloqueia nova consulta caso metricas_teste.csv já exista.


In [ ]:
TEST_CSV = RESULTS / "metricas_teste.csv"

if RUN_FINAL_TEST:
    if TEST_CSV.exists():
        raise RuntimeError("O teste já foi consultado para TPOT. Use o CSV salvo; não repita a avaliação.")
    if not SELECTION_JSON.exists():
        raise FileNotFoundError("Não há budget congelado. Complete primeiro a validação.")
    chosen_budget = json.loads(SELECTION_JSON.read_text(encoding="utf-8"))["budget_selecionado_min"]
    final_model = build_tpot(X_train, y_train, chosen_budget, FINAL_SEED,
                             CHECKPOINTS / f"teste_final_{chosen_budget}m_seed_{FINAL_SEED}")
    metrics, pred = fit_measure_predict(final_model, X_train, y_train, X_test, y_test, LABELS)
    row = {"sistema": "TPOT", "versao": tpot.__version__, "budget_min": chosen_budget,
           "seed": FINAL_SEED, "subconjunto": "test", **metrics}
    pd.DataFrame([row]).to_csv(TEST_CSV, index=False)
    pred.insert(0, "sl_no", test_df.loc[pred.index, "Sl No"].to_numpy())
    pred.to_csv(RESULTS / "predicoes_teste.csv", index_label="row_index")
    (PIPELINES / "pipeline_teste_final.txt").write_text(pipeline_repr(final_model), encoding="utf-8")
    display(pd.DataFrame([row]))
else:
    print("RUN_FINAL_TEST=False. O conjunto de teste não foi consultado.")
